In [1]:
import pandas as pd
import numpy as np
import requests
import os
from pathlib import Path

# Create directories
Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

print("Libraries loaded!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Libraries loaded!
Pandas: 3.0.3
NumPy: 2.4.6


In [2]:
import numpy as np

np.random.seed(42)
n = 2000

diseases = ['Oncology','Cardiovascular','Neurology','Infectious Disease',
            'Diabetes/Metabolic','Respiratory','Immunology/Autoimmune',
            'Rare Disease','Mental Health','Other']
d_w = np.array([0.28,0.14,0.12,0.11,0.09,0.07,0.06,0.06,0.04,0.03])
d_w = d_w/d_w.sum()

phases = ['Phase 1','Phase 2','Phase 3','Phase 4','N/A']
p_w = np.array([0.28,0.35,0.22,0.08,0.07]); p_w = p_w/p_w.sum()

statuses = ['Completed','Recruiting','Active, not recruiting','Terminated','Unknown','Withdrawn']
s_w = np.array([0.38,0.22,0.15,0.12,0.08,0.05]); s_w = s_w/s_w.sum()

interventions = ['Drug','Biological','Device','Behavioral','Procedure','Other']
i_w = np.array([0.52,0.18,0.10,0.08,0.06,0.06]); i_w = i_w/i_w.sum()

sponsors = ['Pfizer','Novartis','Roche','AstraZeneca','Johnson & Johnson',
            'Merck','Bristol-Myers Squibb','Eli Lilly','Sanofi','GSK',
            'Bayer','Boehringer Ingelheim','Amgen','Gilead','AbbVie',
            'University Hospital','NIH','Academic Medical Center',
            'Regional Hospital','Biotech Startup']
sp_w = np.array([0.06]*15 + [0.04,0.04,0.04,0.04,0.14]); sp_w = sp_w/sp_w.sum()

years = np.random.randint(2013, 2025, n)
records = []
for i in range(n):
    year = int(years[i])
    ai_prob = 0.02 + (year - 2013) * 0.015
    phase_val = np.random.choice(phases, p=p_w)
    if phase_val == 'Phase 1': enroll = int(np.random.lognormal(3.5, 0.6))
    elif phase_val == 'Phase 2': enroll = int(np.random.lognormal(4.5, 0.7))
    elif phase_val == 'Phase 3': enroll = int(np.random.lognormal(6.0, 0.8))
    else: enroll = int(np.random.lognormal(5.0, 1.0))
    records.append({
        'nct_id': f'NCT{2013000000+i:08d}',
        'start_year': year,
        'disease_area': np.random.choice(diseases, p=d_w),
        'phase': phase_val,
        'status': np.random.choice(statuses, p=s_w),
        'intervention_type': np.random.choice(interventions, p=i_w),
        'lead_sponsor': np.random.choice(sponsors, p=sp_w),
        'enrollment': min(enroll, 50000),
        'duration_months': min(int(np.random.lognormal(3.0, 0.5)), 120),
        'ai_designed': np.random.random() < ai_prob,
        'multinational': np.random.random() < 0.35,
        'has_biomarker': np.random.random() < 0.42
    })

df = pd.DataFrame(records)
df.to_csv('../data/raw/clinical_trials_data.csv', index=False)
print(f"Generated {len(df)} records")
print(df['disease_area'].value_counts())

Generated 2000 records
disease_area
Oncology                 553
Cardiovascular           305
Neurology                215
Infectious Disease       203
Diabetes/Metabolic       176
Immunology/Autoimmune    152
Respiratory              130
Rare Disease             123
Mental Health             72
Other                     71
Name: count, dtype: int64


In [3]:
print("Dataset Summary")
print("="*40)
print(f"Total trials: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Year range: {df['start_year'].min()} - {df['start_year'].max()}")
print(f"AI-designed: {df['ai_designed'].sum()} ({df['ai_designed'].mean():.1%})")
print(f"\nPhase distribution:")
print(df['phase'].value_counts())

df.to_csv('../data/processed/trials_cleaned.csv', index=False)
print("\nData saved!")

Dataset Summary
Total trials: 2,000
Columns: ['nct_id', 'start_year', 'disease_area', 'phase', 'status', 'intervention_type', 'lead_sponsor', 'enrollment', 'duration_months', 'ai_designed', 'multinational', 'has_biomarker']
Year range: 2013 - 2024
AI-designed: 222 (11.1%)

Phase distribution:
phase
Phase 2    672
Phase 1    553
Phase 3    440
Phase 4    174
N/A        161
Name: count, dtype: int64

Data saved!
